# Transaction Cost Sensitivity

This notebook tests whether the event-based strategy remains robust after realistic cost assumptions. The notebook keeps cost modeling separate from signal generation. The same candidate event universe is repriced under multiple transaction-cost, bid-ask spread, and slippage assumptions.

## Research framing

A residual strategy can look attractive before costs because z-score PnL proxies ignore execution. Every entry and exit consumes spread, commission, and slippage. A robust strategy should not depend on a single optimistic cost assumption or one exact threshold configuration.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.backtest import RobustnessGrid, run_cost_and_threshold_robustness
from src.metrics import cost_adjusted_performance_summary
from src.plotting import plot_transaction_cost_sensitivity, plot_threshold_sensitivity

DATA_DIR = ROOT / 'data' / 'processed'
FIGURE_DIR = ROOT / 'figures'
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Load event labels and model predictions

The preferred inputs are the real event labels and model predictions produced earlier in the pipeline. If those files are not available, the notebook falls back to the placeholder synthetic files included with the package so the workflow can still run end-to-end.

In [ ]:
label_path = DATA_DIR / 'event_labels_table.csv'
prediction_path = DATA_DIR / 'predicted_reversion_probabilities.csv'

if not label_path.exists():
    label_path = DATA_DIR / '18_synthetic_event_labels.csv'
if not prediction_path.exists():
    prediction_path = DATA_DIR / '18_synthetic_predictions.csv'

labels = pd.read_csv(label_path, parse_dates=['event_date', 'exit_date'])
predictions = pd.read_csv(prediction_path)

labels.head(), predictions.head()

## Cost assumptions

The cost scenarios are expressed in residual PnL proxy units because the packaged placeholder backtest does not contain execution-level fills. When real trade prices and hedge weights are available, the same structure should be replaced with dollar or basis-point costs.

In [ ]:
cost_assumptions = [
    {'scenario': 'zero_cost', 'commission_per_trade': 0.000, 'bid_ask_spread_proxy': 0.000, 'slippage': 0.000},
    {'scenario': 'low_cost', 'commission_per_trade': 0.005, 'bid_ask_spread_proxy': 0.010, 'slippage': 0.005},
    {'scenario': 'base_cost', 'commission_per_trade': 0.010, 'bid_ask_spread_proxy': 0.025, 'slippage': 0.015},
    {'scenario': 'stress_cost', 'commission_per_trade': 0.020, 'bid_ask_spread_proxy': 0.060, 'slippage': 0.040},
]

grid = RobustnessGrid(
    entry_thresholds=(1.5, 2.0, 2.5),
    exit_thresholds=(0.25, 0.5, 0.75),
    stop_loss_levels=(2.75, 3.0, 3.5),
    max_holding_periods=(10, 15, 20),
)

## Run sensitivity analysis

The transaction-cost analysis reprices the same trades under different cost assumptions. The threshold analysis changes entry, exit, stop-loss, and holding-period settings as a robustness diagnostic. This is not a claim of live tradability.

In [ ]:
outputs = run_cost_and_threshold_robustness(
    labels,
    predictions,
    cost_assumptions=cost_assumptions,
    grid=grid,
    probability_threshold=0.60,
)

transaction_cost_sensitivity = outputs['transaction_cost_sensitivity']
threshold_sensitivity_table = outputs['threshold_sensitivity_table']
robustness_summary = outputs['robustness_summary']
cost_adjusted_summary = cost_adjusted_performance_summary(transaction_cost_sensitivity)

transaction_cost_sensitivity.head()

In [ ]:
transaction_cost_sensitivity.to_csv(DATA_DIR / 'transaction_cost_sensitivity.csv', index=False)
threshold_sensitivity_table.to_csv(DATA_DIR / 'threshold_sensitivity_table.csv', index=False)
cost_adjusted_summary.to_csv(DATA_DIR / 'cost_adjusted_performance_summary.csv', index=False)
robustness_summary.to_csv(DATA_DIR / 'robustness_summary.csv', index=False)
outputs['transaction_cost_trade_log'].to_csv(DATA_DIR / 'transaction_cost_trade_log.csv', index=False)

## Plots

The cost plot shows how net PnL changes as execution assumptions become less favorable. The threshold plot is a compact robustness view across entry and exit thresholds for the ML-filtered strategy.

In [ ]:
plot_transaction_cost_sensitivity(
    transaction_cost_sensitivity,
    FIGURE_DIR / 'transaction_cost_sensitivity.png',
    metric='net_pnl',
)

plot_threshold_sensitivity(
    threshold_sensitivity_table,
    FIGURE_DIR / 'threshold_sensitivity.png',
    strategy='ml_filtered',
    metric='net_pnl',
)

## Interpretation checks

The key questions are not whether one scenario looks good. The useful checks are whether net PnL collapses under slightly higher costs, whether Sharpe depends on one narrow threshold combination, whether turnover is too high, and whether ML filtering reduces cost drag by avoiding low-quality trades.

In [ ]:
robustness_summary